<a href="https://colab.research.google.com/github/Yufonyuy/Deep_forecasting-USU/blob/main/SAPE_EN_GEE_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAPE-EN: GEE Extraction + Per-Subdepartment LSTM Training
---

**Extracts real satellite climate data from Google Earth Engine for every
sub-department in your GeoJSON, then trains a spatially-aware LSTM that
produces DIFFERENT 6-month drought forecasts for each sub-department.**

## What This Notebook Does

1. **Upload** your `northern_subdepts_clean.geojson`
2. **Extract** real CHIRPS precipitation + ERA5 temperature/PET from GEE
   for each sub-department polygon (2000-2025, monthly)
3. **Compute** elevation and river distance for each sub-department
4. **Engineer** features per sub-department — each time series preserved independently
5. **Train** LSTM with static geographic features (lat, lon, elevation, river dist)
6. **Download** model + scalers + metadata + training data

## Output Files (download all 5)
| File | Drop into |
|------|-----------|
| `lstm_drought_far_north.keras` | `backend/saved_models/` |
| `scaler_X.pkl` | `backend/saved_models/` |
| `scaler_y.pkl` | `backend/saved_models/` |
| `model_metadata.json` | `backend/saved_models/` |
| `per_arrondissement_training_data.csv` | `backend/data/` |

## Time Required
- GEE extraction: ~20-40 minutes (depends on quota, ~312 months × N subdepartments)
- Model training: ~3 minutes on free T4 GPU
- If GEE fails: fallback to model-generated data (~5 min total)

## Step 0: Upload Your GeoJSON

Upload `northern_subdepts_clean.geojson`.
This file contains polygon boundaries for all sub-departments.

In [1]:
from google.colab import files
import json, math

print('📤 Upload northern_subdepts_clean.geojson')
uploaded = files.upload()
fname = list(uploaded.keys())[0]

with open(fname) as f:
    gj = json.load(f)

features = gj['features']
print(f'✅ Loaded {len(features)} features')

# Show sample
for ft in features[:3]:
    p = ft['properties']
    print(f'   {p["sub_department"]:25s} | {p["department"]:20s} | {p["region"]:12s} | area={p["area_sqkm"]:.0f}km²')

# Count regions
from collections import Counter
regions = Counter(ft['properties']['region'] for ft in features)
for r, c in regions.items():
    print(f'   {r}: {c} sub-departments')

📤 Upload northern_subdepts_clean.geojson


Saving northern_subdepts_clean.geojson to northern_subdepts_clean (1).geojson
✅ Loaded 89 features
   Bankim                    | Mayo-Banyo           | Adamawa      | area=2442km²
   Banyo                     | Mayo-Banyo           | Adamawa      | area=5066km²
   Bascheo                   | Bénoué               | North        | area=1049km²
   Adamawa: 21 sub-departments
   North: 21 sub-departments
   Far-North: 47 sub-departments


## Step 1: Install & Authenticate GEE

In [2]:
!pip install -q earthengine-api tensorflow scikit-learn pandas numpy

import ee
import numpy as np
import pandas as pd
import pickle
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'✅ TF {tf.__version__} | GPU: {len(tf.config.list_physical_devices("GPU"))>0}')

✅ TF 2.20.0 | GPU: False


In [4]:
# Authenticate with Google Earth Engine
try:
    ee.Initialize(project="flood-drought-prediction")
    print('✅ Already authenticated with GEE')
except:
    ee.Authenticate()
    ee.Initialize()
    print('✅ GEE authenticated!')

✅ Already authenticated with GEE


## Step 2: Compute Elevation & River Distance Per Sub-Department

These static features are critical — they tell the LSTM WHERE each
sub-department is geographically so it learns spatial climate patterns.

In [5]:
# ── Elevation reference points (SRTM-derived for northern Cameroon) ──
ELEVATION_POINTS = [
    (14.99, 12.08, 295), (14.55, 12.40, 290), (14.71, 12.38, 293),
    (14.30, 10.55, 380), (14.12, 10.05, 420), (15.20, 10.08, 310),
    (13.60, 9.30, 240), (13.88, 9.90, 340), (12.75, 7.40, 1100),
    (13.55, 7.35, 1050), (11.50, 6.25, 700), (13.14, 6.45, 800),
    (14.42, 6.44, 950), (14.96, 6.78, 1050), (12.65, 6.53, 650),
    (11.80, 6.72, 800),
]

# ── River waypoints ──
RIVER_COURSES = {
    "Logone": [(14.55,12.90),(14.65,12.80),(14.80,12.70),(14.90,12.55),
               (15.00,12.40),(15.05,12.20),(15.08,12.05),(15.10,11.88),
               (15.12,11.70),(15.10,11.50),(15.05,11.35),(14.95,11.20),
               (14.88,11.05),(14.80,10.90),(14.70,10.75),(14.60,10.60)],
    "Chari": [(14.40,12.88),(14.50,12.78),(14.65,12.70),(14.85,12.58),
              (15.00,12.50),(15.05,12.30),(15.10,12.15),(15.12,12.00),
              (15.08,11.85),(15.00,11.75),(14.88,11.65),(14.75,11.55)],
    "Mayo_Kebbi": [(13.85,10.38),(14.00,10.42),(14.15,10.48),(14.30,10.55),
                   (14.45,10.65),(14.55,10.75),(14.60,10.85),(14.55,10.95),
                   (14.45,11.00),(14.30,11.00),(14.15,10.95)],
    "Benoue": [(13.15,9.25),(13.30,9.35),(13.45,9.42),(13.60,9.48),
               (13.75,9.55),(13.85,9.68),(13.95,9.82),(14.00,9.95)],
}

def estimate_elevation(lng, lat):
    """IDW interpolation of elevation."""
    weights = []
    for elng, elat, elev in ELEVATION_POINTS:
        d = math.hypot(lng - elng, lat - elat)
        if d < 0.001: return float(elev)
        weights.append((1.0/(d**2 + 1e-6), elev))
    tw = sum(w for w,_ in weights)
    return sum(w*e/tw for w,e in weights)

def dist_to_river(lng, lat):
    """Min distance (km) to nearest river."""
    best = float('inf')
    best_name = 'none'
    for rname, waypoints in RIVER_COURSES.items():
        for rlng, rlat in waypoints:
            d = math.hypot((lng-rlng)*111.32*math.cos(math.radians((lat+rlat)/2)),
                           (lat-rlat)*110.574)
            if d < best: best, best_name = d, rname
    return best, best_name

# Compute static features for each feature
arr_info = []
for ft in features:
    p = ft['properties']
    lat = p.get('center_lat', 10.0)
    lng = p.get('center_lon', 14.5)
    elev = estimate_elevation(lng, lat)
    rdist, rname = dist_to_river(lng, lat)
    arr_info.append({
        'name': p['sub_department'],
        'pcode': p.get('pcode',''),
        'department': p['department'],
        'region': p['region'],
        'lat': lat, 'lng': lng,
        'elevation_m': round(elev, 1),
        'river_dist_km': round(rdist, 1),
        'nearest_river': rname,
        'area_sqkm': p.get('area_sqkm', 0),
        'geometry': ft['geometry'],
    })

print(f'Computed static features for {len(arr_info)} sub-departments')
for a in arr_info[:3]:
    print(f'   {a["name"]:25s} elev={a["elevation_m"]:5.0f}m river={a["nearest_river"]:12s} dist={a["river_dist_km"]:5.0f}km')

Computed static features for 89 sub-departments
   Bankim                    elev=  700m river=Benoue       dist=  378km
   Banyo                     elev=  800m river=Benoue       dist=  317km
   Bascheo                   elev=  363m river=Benoue       dist=   39km


## Step 3: Build GEE FeatureCollection From Polygons

In [6]:
ee_features = []
skipped = 0
for a in arr_info:
    try:
        geom = a['geometry']
        if geom['type'] == 'Polygon':
            ee_geom = ee.Geometry.Polygon(geom['coordinates'])
        elif geom['type'] == 'MultiPolygon':
            ee_geom = ee.Geometry.MultiPolygon(geom['coordinates'])
        else:
            ee_geom = ee.Geometry.Point([a['lng'], a['lat']]).buffer(5000)

        ee_features.append(ee.Feature(ee_geom, {
            'name': a['name'], 'pcode': a['pcode'],
            'department': a['department'], 'region': a['region']
        }))
    except Exception as e:
        skipped += 1
        continue

fc = ee.FeatureCollection(ee_features)
print(f'✅ Built FeatureCollection: {len(ee_features)} sub-departments ({skipped} skipped)')

✅ Built FeatureCollection: 89 sub-departments (0 skipped)


## Step 4: Extract CHIRPS Precipitation (2000-2025)

Uses `reduceRegions` — one GEE call per month, sampling ALL sub-departments at once.
**~312 calls, may take 15-25 minutes.**

In [7]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD')
years = list(range(2000, 2026))
months = list(range(1, 13))

chirps_rows = []
print('Extracting CHIRPS precipitation...')

for year in years:
    for month in months:
        try:
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')
            monthly = chirps.filterDate(start, end).select('precipitation').sum()

            sampled = monthly.reduceRegions(
                collection=fc,
                reducer=ee.Reducer.mean(),
                scale=5566
            )

            results = sampled.getInfo()['features']
            for f in results:
                p = f['properties']
                chirps_rows.append({
                    'year': year, 'month': month,
                    'name': p.get('name', ''),
                    'pcode': p.get('pcode', ''),
                    'department': p.get('department', ''),
                    'region': p.get('region', ''),
                    'precipitation_mm': round(p.get('mean', 0.0) or 0.0, 1)
                })
        except Exception as e:
            print(f'   ⚠ {year}-{month:02d}: {str(e)[:80]}')
            continue
    print(f'   {year} done ({len(chirps_rows)} records)')

chirps_df = pd.DataFrame(chirps_rows)
print(f'\n✅ CHIRPS: {len(chirps_df)} records')
print(f'   Sub-departments: {chirps_df["name"].nunique()}')

Extracting CHIRPS precipitation...
   2000 done (1068 records)
   2001 done (2136 records)
   2002 done (3204 records)
   2003 done (4272 records)
   2004 done (5340 records)
   2005 done (6408 records)
   2006 done (7476 records)
   2007 done (8544 records)
   2008 done (9612 records)
   2009 done (10680 records)
   2010 done (11748 records)
   2011 done (12816 records)
   2012 done (13884 records)
   2013 done (14952 records)
   2014 done (16020 records)
   2015 done (17088 records)
   2016 done (18156 records)
   2017 done (19224 records)
   2018 done (20292 records)
   2019 done (21360 records)
   2020 done (22428 records)
   2021 done (23496 records)
   2022 done (24564 records)
   2023 done (25632 records)
   2024 done (26700 records)
   2025 done (27768 records)

✅ CHIRPS: 27768 records
   Sub-departments: 89


In [11]:
chirps_df.columns.tolist(), chirps_df.head()

(['year',
  'month',
  'name',
  'pcode',
  'department',
  'region',
  'precipitation_mm'],
    year  month     name     pcode  department   region  precipitation_mm
 0  2000      1   Bankim  CM001003  Mayo-Banyo  Adamawa               4.3
 1  2000      1    Banyo  CM001003  Mayo-Banyo  Adamawa               2.6
 2  2000      1  Bascheo  CM006001      Bénoué    North               0.4
 3  2000      1     Beka  CM006002        Faro    North               0.8
 4  2000      1    Belel  CM001005        Vina  Adamawa               0.1)

## Step 5: Extract ERA5 Temperature & PET

Uses ERA5-Land monthly aggregates. Temperature in Kelvin → convert to °C.

In [12]:
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
era5_rows = []
print('Extracting ERA5 temperature + PET...')

for year in years:
    for month in months:
        try:
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')
            img = era5.filterDate(start, end).first()

            # Temperature: Kelvin → Celsius
            t_img = img.select('temperature_2m').subtract(273.15)
            t_sampled = t_img.reduceRegions(
                collection=fc, reducer=ee.Reducer.mean(), scale=11132)

            # PET: potential_evaporation_sum (already positive monthly sum)
            e_img = img.select('potential_evaporation_sum')
            e_sampled = e_img.reduceRegions(
                collection=fc, reducer=ee.Reducer.mean(), scale=11132)

            t_results = t_sampled.getInfo()['features']
            e_results = e_sampled.getInfo()['features']

            for tf, ef in zip(t_results, e_results):
                tp = tf['properties']
                ep = ef['properties']
                era5_rows.append({
                    'year': year, 'month': month,
                    'name': tp.get('name', ''),
                    'pcode': tp.get('pcode', ''),
                    'temperature_mean_c': round(tp.get('mean', 28.0) or 28.0, 1),
                    'pet_mm': round(ep.get('mean', 100.0) or 100.0, 1)
                })
        except Exception as e:
            print(f'   ⚠ {year}-{month:02d}: {str(e)[:80]}')
    print(f'   {year} done ({len(era5_rows)} records)')

era5_df = pd.DataFrame(era5_rows)
print(f'\n✅ ERA5: {len(era5_df)} records')

Extracting ERA5 temperature + PET...


   2000 done (1068 records)
   2001 done (2136 records)
   2002 done (3204 records)


   2003 done (4272 records)
   2004 done (5340 records)
   2005 done (6408 records)
   2006 done (7476 records)
   2007 done (8544 records)
   2008 done (9612 records)
   2009 done (10680 records)
   2010 done (11748 records)
   2011 done (12816 records)
   2012 done (13884 records)
   2013 done (14952 records)
   2014 done (16020 records)
   2015 done (17088 records)
   2016 done (18156 records)
   2017 done (19224 records)
   2018 done (20292 records)
   2019 done (21360 records)
   2020 done (22428 records)


   2021 done (23496 records)
   2022 done (24564 records)
   2023 done (25632 records)
   2024 done (26700 records)
   2025 done (27768 records)

✅ ERA5: 27768 records


## Step 6: Merge All Data + Add Static Features

Joins CHIRPS precipitation with ERA5 temperature/PET.
Adds static geographic features (lat, lon, elevation, river distance)
for each sub-department.

In [13]:
# Merge CHIRPS + ERA5 on name, year, month
df = chirps_df.merge(era5_df, on=['name', 'year', 'month'], how='outer')

# Add date column
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

# Add static features from arr_info
static_lookup = {a['name']: a for a in arr_info}
df['lat'] = df['name'].map(lambda n: static_lookup.get(n, {}).get('lat', 10.0))
df['lng'] = df['name'].map(lambda n: static_lookup.get(n, {}).get('lng', 14.5))
df['elevation_m'] = df['name'].map(lambda n: static_lookup.get(n, {}).get('elevation_m', 350.0))
df['river_dist_km'] = df['name'].map(lambda n: static_lookup.get(n, {}).get('river_dist_km', 50.0))
df['nearest_river'] = df['name'].map(lambda n: static_lookup.get(n, {}).get('nearest_river', 'none'))

df = df.sort_values(['name', 'date'])

print(f'Merged dataset: {len(df)} rows')
print(f'Sub-departments: {df["name"].nunique()}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')

# Per-region summary
for reg in df['region'].unique():
    rdf = df[df['region'] == reg]
    ann = rdf.groupby(['name','year'])['precipitation_mm'].sum().groupby('name').mean()
    print(f'\n{reg}: {rdf["name"].nunique()} sub-deps, '
          f'precip={ann.mean():.0f}mm/yr, temp={rdf["temperature_mean_c"].mean():.1f}°C')

# Save the training data
df.to_csv('per_arrondissement_training_data.csv', index=False)
print(f'\n💾 Saved: per_arrondissement_training_data.csv ({len(df)} rows)')

Merged dataset: 27768 rows
Sub-departments: 89
Date range: 2000-01-01 → 2025-12-01
Columns: ['year', 'month', 'name', 'pcode_x', 'department', 'region', 'precipitation_mm', 'pcode_y', 'temperature_mean_c', 'pet_mm', 'date', 'lat', 'lng', 'elevation_m', 'river_dist_km', 'nearest_river']

Adamawa: 21 sub-deps, precip=1642mm/yr, temp=23.6°C

North: 21 sub-deps, precip=1060mm/yr, temp=28.1°C

Far-North: 47 sub-deps, precip=766mm/yr, temp=28.0°C

💾 Saved: per_arrondissement_training_data.csv (27768 rows)


## Step 7: Feature Engineering Per Sub-Department

**Critical**: Each sub-department's time series is processed independently
to preserve temporal structure. Static features are tiled across all timesteps.

In [14]:
SEQUENCE_LENGTH = 24  # 2 years input
FORECAST_HORIZON = 6  # 6 months output

# Rename for consistency with model
df = df.rename(columns={
    'precipitation_mm': 'precipitation',
    'temperature_mean_c': 'temperature_mean',
    'pet_mm': 'pet'
})

def engineer_features(group_df):
    """Build features for one sub-department's time series."""
    g = group_df.set_index('date').sort_index().copy()

    g['month_sin'] = np.sin(2 * np.pi * g.index.month / 12)
    g['month_cos'] = np.cos(2 * np.pi * g.index.month / 12)

    g['rain_3m'] = g['precipitation'].rolling(3, min_periods=1).sum()
    g['rain_6m'] = g['precipitation'].rolling(6, min_periods=1).sum()
    g['rain_12m'] = g['precipitation'].rolling(12, min_periods=1).sum()

    for col in ['precipitation', 'temperature_mean', 'pet']:
        for lag in [1, 2, 3, 6, 12]:
            g[f'{col}_lag{lag}'] = g[col].shift(lag)

    # SPI-3 and SPI-6
    pm3 = g['precipitation'].rolling(3).mean()
    ltm = g['precipitation'].expanding().mean()
    lts = g['precipitation'].expanding().std()
    g['spi_3'] = (pm3 - ltm) / (lts + 1e-6)
    pm6 = g['precipitation'].rolling(6).mean()
    g['spi_6'] = (pm6 - ltm) / (lts + 1e-6)

    # Static features (tiled across all rows for this sub-department)
    g['static_lat'] = group_df['lat'].iloc[0]
    g['static_lon'] = group_df['lng'].iloc[0]
    g['static_elevation'] = group_df['elevation_m'].iloc[0]
    g['static_river_dist'] = group_df['river_dist_km'].iloc[0]

    return g.dropna()

df_eng = df.groupby('name', group_keys=False).apply(engineer_features)
print(f'Engineered features: {len(df_eng)} rows, {len(df_eng.columns)} columns')

Engineered features: 26700 rows, 41 columns


/tmp/ipykernel_930/1526453067.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_eng = df.groupby('name', group_keys=False).apply(engineer_features)


## Step 8: Build Training Sequences

Each sequence: 24 months of weather data → predicts next 6 months of SPI-3 and SPI-6.
Static features (lat, lon, elevation, river_dist) are included in EVERY timestep.

In [15]:
TIME_FEATURES = [
    'precipitation', 'temperature_mean', 'pet',
    'month_sin', 'month_cos',
    'rain_3m', 'rain_6m', 'rain_12m',
    'precipitation_lag1', 'precipitation_lag2', 'precipitation_lag3',
    'precipitation_lag6', 'precipitation_lag12',
    'temperature_mean_lag1', 'temperature_mean_lag2', 'temperature_mean_lag3',
    'temperature_mean_lag6', 'temperature_mean_lag12',
    'pet_lag1', 'pet_lag2', 'pet_lag3',
    'pet_lag6', 'pet_lag12',
]
STATIC_FEATURES = ['static_lat', 'static_lon', 'static_elevation', 'static_river_dist']
TARGETS = ['spi_3', 'spi_6']

available_time = [c for c in TIME_FEATURES if c in df_eng.columns]
available_static = [c for c in STATIC_FEATURES if c in df_eng.columns]
available_targets = [c for c in TARGETS if c in df_eng.columns]

X, y = [], []
for arr_name, group in df_eng.groupby('name'):
    group = group.sort_index()
    n = len(group)
    if n < SEQUENCE_LENGTH + FORECAST_HORIZON:
        continue

    t_vals = group[available_time].values
    s_vals = group[available_static].values[0]  # same for all timesteps
    y_vals = group[available_targets].values

    for i in range(n - SEQUENCE_LENGTH - FORECAST_HORIZON + 1):
        xt = t_vals[i:i+SEQUENCE_LENGTH]
        xs = np.tile(s_vals, (SEQUENCE_LENGTH, 1))
        X.append(np.concatenate([xt, xs], axis=1))
        y.append(y_vals[i+SEQUENCE_LENGTH:i+SEQUENCE_LENGTH+FORECAST_HORIZON])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
print(f'X: {X.shape} (samples × timesteps × features)')
print(f'y: {y.shape} (samples × horizon × targets)')
print(f'Features: {len(available_time)} temporal + {len(available_static)} static = {X.shape[2]} total')

X: (24119, 24, 27) (samples × timesteps × features)
y: (24119, 6, 2) (samples × horizon × targets)
Features: 23 temporal + 4 static = 27 total


## Step 9: Train/Validation Split & Scale

In [16]:
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))
train_idx, val_idx = indices[:split], indices[split:]

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

# Scale features
scaler_X = StandardScaler()
shp = X_train.shape
X_train = scaler_X.fit_transform(X_train.reshape(-1, shp[-1])).reshape(shp)
X_val = scaler_X.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)

# Scale targets
scaler_y = StandardScaler()
yshp = y_train.shape
y_train = scaler_y.fit_transform(y_train.reshape(-1, yshp[-1])).reshape(yshp)
y_val = scaler_y.transform(y_val.reshape(-1, y_val.shape[-1])).reshape(y_val.shape)

with open('scaler_X.pkl', 'wb') as f: pickle.dump(scaler_X, f)
with open('scaler_y.pkl', 'wb') as f: pickle.dump(scaler_y, f)

print(f'Train: X={X_train.shape}, y={y_train.shape}')
print(f'Val:   X={X_val.shape}, y={y_val.shape}')
print('✅ Scalers saved')

Train: X=(19295, 24, 27), y=(19295, 6, 2)
Val:   X=(4824, 24, 27), y=(4824, 6, 2)
✅ Scalers saved


## Step 10: Build & Train LSTM

In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Reshape
from tensorflow.keras.optimizers import Adam

seq_len, n_feat = X_train.shape[1], X_train.shape[2]
hz, nt = y_train.shape[1], y_train.shape[2]

print(f'Input:  ({seq_len} timesteps × {n_feat} features)')
print(f'Output: ({hz} months × {nt} targets — SPI-3, SPI-6)')

model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(seq_len, n_feat),
         recurrent_dropout=0.15),
    BatchNormalization(),
    Dropout(0.25),
    LSTM(64, return_sequences=False, recurrent_dropout=0.15),
    BatchNormalization(),
    Dropout(0.25),
    Dense(48, activation='relu'),
    Dropout(0.15),
    Dense(24, activation='relu'),
    Dense(hz * nt),
    Reshape((hz, nt))
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

Input:  (24 timesteps × 27 features)
Output: (6 months × 2 targets — SPI-3, SPI-6)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 24, 128)        │        79,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 24, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 48)             │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 24)             │         1,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 12)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 6, 2)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,644 (525.95 KB)

 Trainable params: 134,260 (524.45 KB)

 Non-trainable params: 384 (1.50 KB)

In [19]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 35s 178ms/step - loss: 0.3510 - mae: 0.4506 - val_loss: 0.3970 - val_mae: 0.5382 - learning_rate: 0.0010
Epoch 2/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 38s 158ms/step - loss: 0.1463 - mae: 0.2983 - val_loss: 0.1276 - val_mae: 0.2857 - learning_rate: 0.0010
Epoch 3/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 25s 167ms/step - loss: 0.1181 - mae: 0.2653 - val_loss: 0.0727 - val_mae: 0.1990 - learning_rate: 0.0010
Epoch 4/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 25s 167ms/step - loss: 0.1045 - mae: 0.2488 - val_loss: 0.0609 - val_mae: 0.1819 - learning_rate: 0.0010
Epoch 5/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 25s 169ms/step - loss: 0.0956 - mae: 0.2370 - val_loss: 0.0537 - val_mae: 0.1705 - learning_rate: 0.0010
Epoch 6/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 41s 170ms/step - loss: 0.0901 - mae: 0.2294 - val_loss: 0.0528 - val_mae: 0.1664 - learning_rate: 0.0010
Epoch 7/60
151/151 ━━━━━━━━━━━━━━━━━━━━ 26s 172ms/step - loss: 0.0858 - mae: 0.2237 - val_loss: 0.0485 - val_mae: 0.1595 - learnin

## Step 11: Evaluate

In [ ]:
y_pred = model.predict(X_val, verbose=0)
y_pred_flat = scaler_y.inverse_transform(y_pred.reshape(-1, nt))
y_val_flat = scaler_y.inverse_transform(y_val.reshape(-1, nt))

mse = mean_squared_error(y_val_flat, y_pred_flat)
mae = mean_absolute_error(y_val_flat, y_pred_flat)
print(f'Validation MSE: {mse:.4f}')
print(f'Validation MAE: {mae:.4f}')

for t in range(nt):
    yp = y_pred_flat[:, t]
    yt = y_val_flat[:, t]
    print(f'  {TARGETS[t]}: MSE={mean_squared_error(yt, yp):.4f}, MAE={mean_absolute_error(yt, yp):.4f}')

## Step 12: Verify Spatial Awareness

The model must produce DIFFERENT predictions for different sub-departments.
If all predictions are identical, the static features aren't being used.

In [20]:
test_names = [a['name'] for a in arr_info[:5]]
predictions = {}

for arr_name in test_names:
    group = df[df['name'] == arr_name]
    if len(group) == 0:
        continue

    g = engineer_features(group)
    if len(g) < SEQUENCE_LENGTH:
        continue

    t_vals = g[available_time].iloc[-SEQUENCE_LENGTH:].values
    s_vals = g[available_static].iloc[0].values
    xs = np.tile(s_vals, (SEQUENCE_LENGTH, 1))
    X_input = np.concatenate([t_vals, xs], axis=1)
    X_scaled = scaler_X.transform(X_input).reshape(1, SEQUENCE_LENGTH, -1)

    pred = model.predict(X_scaled, verbose=0)
    pred_act = scaler_y.inverse_transform(pred.reshape(-1, nt))

    region = group['region'].iloc[0]
    spi3 = [round(float(v), 3) for v in pred_act[:, 0]]
    predictions[arr_name] = {'region': region, 'spi3': spi3}

    risk = ('🔴 EXTREME' if spi3[0] < -1.5 else '🟠 SEVERE' if spi3[0] < -1.0
            else '🟡 MODERATE' if spi3[0] < -0.5 else '🟢 NORMAL')
    print(f'{arr_name:25s} ({region:12s}) → SPI-3: {spi3} {risk}')

spi3_vals = [v['spi3'][0] for v in predictions.values()]
unique = len(set(round(s, 4) for s in spi3_vals))
print(f'\n✅ {unique}/{len(predictions)} unique SPI-3 values → model IS spatially-aware!')
if unique < 2:
    print('⚠ WARNING: All predictions identical! Static features may not be working.')

Bankim                    (Adamawa     ) → SPI-3: [-0.982, -0.984, -0.862, -0.546, -0.091, 0.333] 🟡 MODERATE
Banyo                     (Adamawa     ) → SPI-3: [-0.989, -0.999, -0.854, -0.477, 0.002, 0.421] 🟡 MODERATE
Bascheo                   (North       ) → SPI-3: [-0.862, -0.86, -0.783, -0.572, -0.201, 0.248] 🟡 MODERATE
Beka                      (North       ) → SPI-3: [-0.906, -0.927, -0.837, -0.559, -0.125, 0.338] 🟡 MODERATE
Belel                     (Adamawa     ) → SPI-3: [-0.989, -0.998, -0.878, -0.555, -0.088, 0.365] 🟡 MODERATE

✅ 4/5 unique SPI-3 values → model IS spatially-aware!


## Step 13: Save & Download Everything

In [22]:
import json
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Calculate validation MSE and MAE
y_pred = model.predict(X_val, verbose=0)
y_pred_flat = scaler_y.inverse_transform(y_pred.reshape(-1, nt))
y_val_flat = scaler_y.inverse_transform(y_val.reshape(-1, nt))

mse = mean_squared_error(y_val_flat, y_pred_flat)
mae = mean_absolute_error(y_val_flat, y_pred_flat)

# Save model
model.save('lstm_drought_far_north.keras')
print('✅ Model saved')

# Save metadata
all_features = available_time + available_static
metadata = {
    'model_version': '2.0.0',
    'trained_at': datetime.now().isoformat(),
    'sequence_length': SEQUENCE_LENGTH,
    'forecast_horizon': FORECAST_HORIZON,
    'n_targets': nt,
    'target_names': available_targets,
    'feature_names': all_features,
    'time_features': available_time,
    'static_features': available_static,
    'n_arrondissements': df['name'].nunique(),
    'training_samples': len(X),
    'val_mse': float(mse),
    'val_mae': float(mae),
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'✅ Metadata: {len(all_features)} features, {len(X)} training sequences')

print('\n📥 DOWNLOADING 5 FILES...')
print('='*50)
files.download('lstm_drought_far_north.keras')
files.download('scaler_X.pkl')
files.download('scaler_y.pkl')
files.download('model_metadata.json')
files.download('per_arrondissement_training_data.csv')

print('\n' + '='*60)
print('✅ COMPLETE!')
print('='*60)
print('\n📋 Place these files in your project:')
print('  lstm_drought_far_north.keras         → backend/saved_models/')
print('  scaler_X.pkl                         → backend/saved_models/')
print('  scaler_y.pkl                         → backend/saved_models/')
print('  model_metadata.json                  → backend/saved_models/')
print('  per_arrondissement_training_data.csv → backend/data/')
print('\nThen run: python run.py api')

✅ Model saved
✅ Metadata: 27 features, 24119 training sequences

📥 DOWNLOADING 5 FILES...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ COMPLETE!

📋 Place these files in your project:
  lstm_drought_far_north.keras         → backend/saved_models/
  scaler_X.pkl                         → backend/saved_models/
  scaler_y.pkl                         → backend/saved_models/
  model_metadata.json                  → backend/saved_models/
  per_arrondissement_training_data.csv → backend/data/

Then run: python run.py api


---
## ⚡ FALLBACK: Model-Generated Training Data

If GEE authentication fails or you hit quota limits, run this cell instead of Steps 1-6.
It generates realistic per-subdepartment training data using geographic models.
Then continue from Step 7.

In [ ]:
# ⚡ FALLBACK: Generate training data without GEE
print('⚡ Generating model-based training data (no GEE needed)...')

np.random.seed(42)

REGION_CLIMATE = {
    "Far-North": {"annual_precip_mm": 650, "temp_mean_c": 28.5, "temp_range_c": 8.0, "pet_annual_mm": 1800},
    "North": {"annual_precip_mm": 1050, "temp_mean_c": 27.0, "temp_range_c": 6.5, "pet_annual_mm": 1600},
    "Adamawa": {"annual_precip_mm": 1500, "temp_mean_c": 24.0, "temp_range_c": 5.0, "pet_annual_mm": 1300},
}
MONTHLY_RAIN_FRAC = {1:0.00,2:0.00,3:0.01,4:0.04,5:0.10,6:0.15,7:0.22,8:0.25,9:0.17,10:0.05,11:0.01,12:0.00}

rows = []
for a in arr_info:
    lat, lng = a['lat'], a['lng']
    climate = REGION_CLIMATE.get(a['region'], REGION_CLIMATE['North'])
    lat_factor = max(0.0, min(1.5, (lat - 7.0) / 6.0))
    elev_factor = max(-0.2, min(0.5, (a['elevation_m'] - 300) / 800))
    river_factor = 0.08 if a['river_dist_km'] < 30 else (0.03 if a['river_dist_km'] < 60 else 0.0)

    annual_precip = climate['annual_precip_mm'] * (1.0 - lat_factor*0.45 + elev_factor*0.30 + river_factor)
    annual_precip = max(350, min(2000, annual_precip))
    temp_mean = climate['temp_mean_c'] + lat_factor*2.5 - elev_factor*3.0
    temp_mean = max(22.0, min(30.0, temp_mean))
    temp_range = climate['temp_range_c'] + lat_factor*1.5
    annual_pet = climate['pet_annual_mm'] * (1.0 + lat_factor*0.30 - elev_factor*0.20)
    annual_pet = max(1000, min(2200, annual_pet))

    for year in range(2000, 2026):
        year_mod = 1.0 + 0.15*math.sin(2*math.pi*(year-2000)*2.5/26 + lat*0.3)
        for month in range(1, 13):
            rain = annual_precip * MONTHLY_RAIN_FRAC[month] * year_mod
            precip = max(0, rain + np.random.normal(0, rain*0.2)) if rain > 0 else 0
            seas_temp = temp_mean + (temp_range/2)*math.sin(2*math.pi*(month-4)/12)
            temp = seas_temp + np.random.normal(0, 1.0)
            pet_base = annual_pet/12
            pet = max(20, pet_base*(1.0+0.4*math.sin(2*math.pi*(month-3)/12)) + np.random.normal(0, pet_base*0.12))
            rows.append({
                'date': f'{year}-{month:02d}-01', 'year': year, 'month': month,
                'name': a['name'], 'pcode': a['pcode'],
                'department': a['department'], 'region': a['region'],
                'precipitation_mm': round(precip, 1),
                'temperature_mean_c': round(temp, 1), 'pet_mm': round(pet, 1),
                'lat': lat, 'lng': lng,
                'elevation_m': a['elevation_m'], 'river_dist_km': a['river_dist_km'],
                'nearest_river': a['nearest_river'],
            })

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
print(f'Generated {len(df)} rows for {df["name"].nunique()} sub-departments')
df.to_csv('per_arrondissement_training_data.csv', index=False)
print('Now continue from Step 7.')